# 10 — Latin-Script Controls and the DEU SBI Reversal (Table 16, Appendix E)

The Indic results could have a trivial explanation: non-Latin script is simply
harder. The WMT24 controls rule that out.

**ENG-SPA** anchors the Parity zone (IPI = 0.009) and shows what an encoder
handling a language well looks like. **ENG-DEU** uses the same Latin alphabet
yet exhibits IP suppression of the same magnitude as the Indic native range —
it lands in Burden. Fragmentation is therefore driven by how the encoder's
vocabulary is allocated, not by script surface.

Appendix E adds a second, sharper contrast: in DEU the SBI–severity relation
runs the **opposite** way to the Indic case. Error-free German segments carry a
*higher* SBI than badly translated ones, because inadequate MT output tends to be
shorter and more generic and therefore fragments less. SBI remains a valid zone
diagnostic in both regimes, but its correlation with quality is not universal.

**Input:** `../data/latin/wmt24_ende_enes_metrics.xlsx`
**Output:** `../results/tables/latin_controls_full.csv`

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)

## Loading the WMT24 Sheets

In [ ]:
from scipy import stats

PAIRS = {"DEU": "German", "SPA": "Spanish"}
latin = {iso: pd.read_excel(DATA_LATIN, sheet_name=sheet) for iso, sheet in PAIRS.items()}
for iso, d in latin.items():
    print(f"  ENG-{iso}: {len(d):,} rows, {d['system'].nunique()} systems")

## IP, TP, SBI and Zone Placement

Same definitions as the Indic side: IP and TP on the target column, SBI as the
mean per-segment TP/IP ratio, IPI as distance from parity.

In [ ]:
def zone(ipi):
    if ipi < 0.05:
        return "Parity"
    if ipi > 0.70:
        return "Paradox"
    return "Burden"


rows = []
print(f"{'Pair':>9}  {'rows':>6}  {'IP':>7}  {'TP':>7}  {'SBI':>6}  {'IPI':>7}  "
      f"{'zone':>8}  {'IP-COMET r':>11}")
print("-" * 74)
for iso in ["SPA", "DEU"]:
    d = latin[iso]
    ip = pd.to_numeric(d["target_xlmr_IP"], errors="coerce")
    tp = pd.to_numeric(d["target_xlmr_TP"], errors="coerce")
    cm = pd.to_numeric(d["COMET"], errors="coerce")
    ok = ip.notna() & cm.notna()
    r = stats.pearsonr(ip[ok], cm[ok])[0]
    ipi = abs(ip.mean() - 1)
    rows.append(dict(pair=f"ENG-{iso}", rows=len(d), ip=ip.mean(), tp=tp.mean(),
                     sbi=(tp / ip).mean(), ipi=ipi, zone=zone(ipi), ip_comet_r=r))
    print(f"{'ENG-' + iso:>9}  {len(d):>6}  {ip.mean():>7.3f}  {tp.mean():>7.3f}  "
          f"{(tp / ip).mean():>6.2f}  {ipi:>7.3f}  {zone(ipi):>8}  {r:>+11.3f}")

latin_tbl = pd.DataFrame(rows).set_index("pair")

# ── Cross-verification against Tables 13 and 16 ──────────────────────────────
assert abs(latin_tbl.loc["ENG-SPA", "ip"] - 1.009) < 0.001
assert abs(latin_tbl.loc["ENG-DEU", "ip"] - 0.535) < 0.001
assert abs(latin_tbl.loc["ENG-SPA", "sbi"] - 1.12) < 0.005
assert abs(latin_tbl.loc["ENG-DEU", "sbi"] - 2.24) < 0.005
assert abs(latin_tbl.loc["ENG-SPA", "ipi"] - 0.009) < 0.001
assert abs(latin_tbl.loc["ENG-DEU", "ipi"] - 0.465) < 0.001
assert abs(latin_tbl.loc["ENG-DEU", "ip_comet_r"] - (-0.262)) < 0.001
assert abs(latin_tbl.loc["ENG-SPA", "ip_comet_r"] - (-0.021)) < 0.001
assert latin_tbl.loc["ENG-SPA", "zone"] == "Parity"
assert latin_tbl.loc["ENG-DEU", "zone"] == "Burden"
print("\n\u2713 ENG-SPA: IP = 1.009, SBI = 1.12, IPI = 0.009 \u2192 Parity")
print("\u2713 ENG-DEU: IP = 0.535, SBI = 2.24, IPI = 0.465 \u2192 Burden")
print("\u2713 IP\u2013COMET Pearson diverges by a factor of twelve "
      "(DEU -0.262 vs SPA -0.021)")

## DEU Sits Inside the Indic Native Range

The refutation of the script-surface explanation, stated numerically.

In [ ]:
indic_ip = {l: full_ip for l, full_ip in zip(
    LANG_ORDER,
    [pd.read_excel(DATA_XLMR, sheet_name=SHEET_MAP[l])[COL_IP_NAT].mean()
     for l in LANG_ORDER])}
lo, hi = min(indic_ip.values()), max(indic_ip.values())
deu_ip = latin_tbl.loc["ENG-DEU", "ip"]
print(f"  Indic native IP range : {lo:.3f} (GUJ) \u2013 {hi:.3f} (HIN)")
print(f"  ENG-DEU IP            : {deu_ip:.3f}")
print(f"  ENG-SPA IP            : {latin_tbl.loc['ENG-SPA', 'ip']:.3f}")

assert lo <= deu_ip <= hi, "DEU should fall inside the Indic native IP range"
print(f"\n\u2713 ENG-DEU ({deu_ip:.3f}) lies inside the Indic native range "
      f"[{lo:.3f}, {hi:.3f}] despite using the Latin alphabet")
print("  Script surface therefore does not explain the burden; vocabulary "
      "allocation does.")

## The SBI Reversal (Appendix E)

Per-severity SBI for ENG-DEU. In the Indic setting fragmentation rises with
error severity; here it falls.

In [ ]:
d = latin["DEU"].copy()
d["SBI"] = (pd.to_numeric(d["target_xlmr_TP"], errors="coerce")
            / pd.to_numeric(d["target_xlmr_IP"], errors="coerce"))

print("ENG-DEU mean SBI by annotated severity")
print(f"{'Severity':>10}  {'N':>6}  {'mean SBI':>9}")
print("-" * 29)
groups = {}
for sev in ["No-error", "minor", "major"]:
    g = d[d["severity"] == sev]["SBI"].dropna()
    groups[sev] = g
    print(f"{sev:>10}  {len(g):>6}  {g.mean():>9.3f}")

H, p = stats.kruskal(*groups.values())
print(f"\n  Kruskal\u2013Wallis H = {H:.1f}   p = {p:.1e}")

assert abs(groups["No-error"].mean() - 2.349) < 0.005
assert abs(groups["major"].mean() - 2.129) < 0.005
assert abs(H - 135.0) < 0.5
assert groups["No-error"].mean() > groups["major"].mean()
print(f"\n\u2713 Error-free SBI = {groups['No-error'].mean():.3f}, "
      f"major-error SBI = {groups['major'].mean():.3f} (paper: 2.349 / 2.129)")
print(f"\u2713 Kruskal\u2013Wallis H = {H:.1f} (paper: 135.0)")
print("\u2713 Direction is OPPOSITE to the romanised-Indic case: in DEU, worse "
      "output fragments LESS")

## MATTR and Byte Premium for the Controls

Two vocabulary-free measures, computed the same way as on the Indic side:
MATTR over case-folded whitespace tokens with a 500-token window, and byte
premium as the mean UTF-8 byte length per word in the target divided by the same
quantity in the English source.

German exceeds Spanish on MATTR, consistent with the compound-noun mechanism,
while the two byte premiums are close to each other — so UTF-8 encoding overhead
does not account for the IP gap between the two pairs.

In [ ]:
def mattr(texts, window=500):
    # Moving-average type-token ratio over case-folded whitespace tokens.
    toks = []
    for t in texts:
        if pd.isna(t):
            continue
        toks.extend(str(t).lower().split())
    if len(toks) < window:
        return len(set(toks)) / max(len(toks), 1)
    return float(np.mean([len(set(toks[i:i + window])) / window
                          for i in range(len(toks) - window + 1)]))


def bytes_per_word(texts):
    # Mean UTF-8 byte length over individual whitespace-delimited words.
    lengths = [len(w.encode("utf-8")) for t in texts if pd.notna(t)
               for w in str(t).split()]
    return float(np.mean(lengths))


print(f"{'Pair':>9}  {'MATTR':>7}  {'byte premium':>13}")
print("-" * 33)
extra = {}
for iso in ["SPA", "DEU"]:
    d = latin[iso]
    m = mattr(d["target"].tolist())
    bp = bytes_per_word(d["target"]) / bytes_per_word(d["source"])
    extra[iso] = (m, bp)
    print(f"{'ENG-' + iso:>9}  {m:>7.3f}  {bp:>13.3f}")

assert abs(extra["DEU"][0] - 0.634) < 0.001
assert abs(extra["SPA"][0] - 0.581) < 0.001
assert abs(extra["DEU"][1] - 1.25) < 0.005
assert abs(extra["SPA"][1] - 1.09) < 0.005
print("\n\u2713 MATTR: DEU 0.634 > SPA 0.581 (Table 17)")
print("\u2713 Byte premium: DEU 1.25 vs SPA 1.09 \u2014 close to each other, so")
print("  encoding overhead does not account for the IP gap (0.535 vs 1.009)")

## Saving Results

In [ ]:
sev = pd.DataFrame({
    "severity": list(groups), "N": [len(g) for g in groups.values()],
    "mean_sbi": [g.mean() for g in groups.values()],
})
sev["kruskal_H"] = H
sev["kruskal_p"] = p

path = TABLES_DIR / "latin_controls_full.csv"
latin_tbl.to_csv(path)
sev.to_csv(TABLES_DIR / "deu_sbi_severity.csv", index=False)
print(latin_tbl.round(3).to_string())
print()
print(sev.round(3).to_string(index=False))
print(f"\nSaved \u2192 {path}")
print(f"Saved \u2192 {TABLES_DIR / 'deu_sbi_severity.csv'}")
print("\n=== Notebook 10 — output manifest ===")
print("  latin_controls_full.csv")
print("  deu_sbi_severity.csv")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1